# 08 · Caso de clase: del rechazo al dato explicable

**Objetivo:** integrar el recorrido mediante una simulación breve en memoria, no un pipeline desplegable.

**Modalidad:** material para explicar en clase. Los ejemplos se ejecutan únicamente en memoria, con datos ficticios. No conectan a una base, no crean archivos ni ejecutan una ingesta. Los bloques SQL son ilustrativos y no se envían a ningún servidor.

**Prerrequisitos:** Python con pandas. Cada notebook es independiente: no requiere scripts `.py` ni haber ejecutado otra notebook.


In [1]:
# Importar conexión centralizada desde el módulo
import sys
sys.path.insert(0, '..')

from conexion_sqlserver import obtener_engines, SERVIDOR, BD_ORIGEN, BD_DESTINO, USUARIO

# Obtener los engines
engine_origen, engine_destino = obtener_engines()
print(f'Conexiones establecidas correctamente.')
print(f'Servidor: {SERVIDOR}')
print(f'Origen: {BD_ORIGEN}')
print(f'Destino: {BD_DESTINO}')


Conexion OK (origen): WideWorldImporters
Conexion OK (destino): WideWorldImportersDW2
Conexiones establecidas correctamente.
Servidor: localhost\SQLEXPRESS02
Origen: WideWorldImporters
Destino: WideWorldImportersDW2


## Guion del docente

1. Recibimos dos detalles ficticios.
2. Uno tiene referencia e importe inválidos.
3. Registramos una fila en cuarentena con dos motivos.
4. El Steward documenta una corrección.
5. Revalidamos, calculamos medidas y revisamos el grano.
6. Explicamos qué se publicaría y qué garantías faltan fuera de la simulación.


In [ ]:
from copy import deepcopy
from decimal import Decimal, InvalidOperation
import pandas as pd

raw = [
    {"id": 1, "producto_nk": 999, "total": "ERROR", "impuesto": "4.20"},
    {"id": 2, "producto_nk": 20, "total": "-6.05", "impuesto": "-1.05"},
]
productos = {10: 201, 20: 202}

def evaluar(filas):
    ids = pd.Series([r["id"] for r in filas])
    duplicados = set(ids[ids.duplicated(keep=False)])
    aceptadas, rechazadas = [], []
    for r in filas:
        razones = []
        if r["id"] in duplicados:
            razones.append("ID_DUPLICADO")
        if r["producto_nk"] not in productos:
            razones.append("PRODUCTO_INEXISTENTE")
        for campo in ["total", "impuesto"]:
            try:
                valor = Decimal(r[campo])
                if not valor.is_finite() or valor != valor.quantize(Decimal("0.01")):
                    razones.append("DECIMAL_INVALIDO_" + campo.upper())
            except (InvalidOperation, ValueError):
                razones.append("DECIMAL_INVALIDO_" + campo.upper())
        if razones:
            rechazadas.append({"id": r["id"], "motivos": razones})
        else:
            aceptadas.append(deepcopy(r))
    return aceptadas, rechazadas

aceptadas, cuarentena = evaluar(raw)
display(pd.DataFrame(cuarentena))
print("Entrada:", len(raw), "Aceptadas:", len(aceptadas), "Cuarentena:", len(cuarentena))
print("Decisión inicial:", "BLOQUEAR" if cuarentena else "APTO PARA REVISIÓN")


In [ ]:
correccion = {
    "id": 1, "actor": "steward_demo",
    "motivo": "Valores confirmados por ficha ficticia del ejercicio",
    "antes": deepcopy(raw[0]),
    "cambios": {"producto_nk": 10, "total": "24.20"},
}
candidatas = deepcopy(raw)
candidatas[0].update(correccion["cambios"])
correccion["despues"] = deepcopy(candidatas[0])
aceptadas, cuarentena = evaluar(candidatas)
display(pd.DataFrame([correccion]))
print("Cuarentena tras revisión:", cuarentena)
print("RAW conserva:", raw[0])


In [ ]:
if cuarentena:
    raise ValueError("Resolver los rechazos antes de mostrar los hechos propuestos")

propuesta = []
for fila in aceptadas:
    total, impuesto = Decimal(fila["total"]), Decimal(fila["impuesto"])
    propuesta.append({"linea_id": fila["id"], "producto_sk": productos[fila["producto_nk"]],
                      "neto": total - impuesto, "impuesto": impuesto, "total": total})
display(pd.DataFrame(propuesta))
totales = {campo: sum((r[campo] for r in propuesta), Decimal("0.00"))
           for campo in ["neto", "impuesto", "total"]}
print("Totales propuestos:", totales)
print("No se escribió en ninguna base ni archivo.")


## Cierre: qué demostró y qué no

El ejercicio muestra rechazo, múltiples motivos, corrección auditada,
revalidación, búsqueda de una SK y cálculo de medidas. El resultado propuesto
tiene dos líneas: neto **15.00**, impuesto **3.15**, total **18.15**.

No demuestra transacciones, permisos, persistencia, extracción consistente,
idempotencia de una base real ni historia SCD2. Esos temas se explican en
00–07. No hay jobs, CLI, servicios, orquestador ni automatización de carga.


## Actividad de discusión

Cambie el total corregido a un valor inválido y vuelva a evaluar. ¿Por qué no alcanza con registrar al actor? **Respuesta:** la autorización no reemplaza la validación del contenido.